# Türkiye Üniversiteleri Arası Bilimsel İş Birliği Ağı
## Sosyal Ağ Analizi — Final Raporu ve İleri Analiz Hattı (30 Ekim 2026)

**Kurum:** Manisa Celal Bayar Üniversitesi — Büyük Veri Analistliği  
**Dönem:** 2026–2027 Güz / Sosyal Ağ Analizi  
**Proje Kapsamı:** OpenAlex 2020–2025 dönemi Türkiye Bilgisayar Bilimi (Computer Science) yayınları üzerinden kurumsal iş birliği ağı.

> **Önemli Yeniden Üretilebilirlik Notu:**  
> Bu notebook, 30 Ekim 2026 final teslim ödev kartında tanımlanan tüm ileri analiz adımlarını ve üretilen çıktıları interaktif olarak incelemek, doğrulamak ve görselleştirmek amacıyla tasarlanmıştır.  
> Analiz hattının tamamını (100 rastgele ağ içeren Null model ve ayrık Negatif Binom regresyonu dahil) sıfırdan terminalde çalıştırmak için ana giriş noktası **`scripts/run_final_analysis.py`** betiğidir:
> ```bash
> python scripts/run_final_analysis.py
> ```

### Notebook Analiz Akışı:
1. **Veri Yükleme ve Ağ Grafiği:** Kapsam B1 (186 aktif üniversite, tek bileşen) ve Kapsam B (194 üniversite, duyarlılık analizi).
2. **Temel Ağ Ölçüleri ve Merkeziyetler:** 5 merkeziyet ölçüsü (Degree, Weighted Degree, Betweenness, Closeness, PageRank) ve Spearman korelasyonu.
3. **Derece Dağılımı ve Güç Yasası:** `powerlaw` ile log-log CCDF grafiği, $\alpha, x_{min}$, lognormal ve üstel model karşılaştırmaları.
4. **Louvain ve Leiden Topluluk Tespiti:** 10 farklı sabit tohum ile ortak NetworkX modülerliği, NMI/ARI kararlılık analizi ve en büyük topluluk profilleri.
5. **Null Model Analizi:** Derece dizisini %100 koruyan 100 rastgele ağ ile ağırlıksız topolojik karşılaştırma, $C, L, Q$ Z-skorları ve Small-Worldness ($\\sigma$).
6. **Zorunlu İleri Yöntem: Çekim (Gravity) Modeli Regresyonu:** 17.205 ikili çift, Haversine mesafesi ($d_{eff} = d + 1\text{ km}$), Poisson ve veriden $\\alpha$ parametresi tahmin edilen ayrık Negative Binomial (NB2) GLM modelleri, aşırı yayılım kontrolü ve Ege Bölgesi çekim etkisi.
7. **Etkileşimli Coğrafi Harita:** Folium ile Türkiye geneli iş birliği haritası ve açısal dağıtım (jitter).

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image

# Proje kök dizinini ve veri yollarını hem proje kökünden hem notebooks/ dizininden çalışabilecek şekilde sağlamlaştır
PROJECT_ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, "data/processed")
REPORTS_GEN_DIR = os.path.join(PROJECT_ROOT, "reports/generated")

from scripts.run_final_analysis import load_network_data, build_scope_graph
from scripts.analysis_metrics import compute_network_topology, compute_centralities
from scripts.analysis_powerlaw import fit_powerlaw_distribution
from scripts.analysis_community import analyze_communities, SEEDS_10
from scripts.analysis_null_model import run_null_model_experiments
from scripts.analysis_gravity import build_dyadic_dataset, fit_gravity_models
from scripts.visualize_map import generate_collaboration_map

print("Gerekli kütüphaneler ve analiz modülleri başarıyla yüklendi.")

Gerekli kütüphaneler ve analiz modülleri başarıyla yüklendi.


## 1. Veri Yükleme ve Ağ Grafiği İnşası
İşlenmiş kurum metaverileri (`institutions.csv`) ve kenar listesi (`edges.csv`) yüklenir.  
- **Kapsam B1 (Birincil Kapsam):** 186 kural tabanlı aktif üniversite. Ağ tek bir bağlı bileşenden oluşur ve izole düğüm içermez ($N=186$, $M=3.848$, Density=$0.2237$).  
- **Kapsam B (Duyarlılık Kapsamı):** 194 üniversite (186 aktif + 7 KHK ve 1 CBK ile kapatılmış tarihsel kurum). Canik Başarı Üniversitesi izole düğümdür ($N=194$, $M=3.887$, Density=$0.2076$).

In [2]:
df_inst, df_edges = load_network_data(
    os.path.join(DATA_DIR, "institutions.csv"),
    os.path.join(DATA_DIR, "edges.csv")
)

# Kapsam B1: 186 Aktif Üniversite
G_b1, df_b1 = build_scope_graph(df_inst, df_edges, allowed_categories=["active_university"])

# Kapsam B: 194 Aktif ve Kapatılan Üniversite
G_b, df_b = build_scope_graph(df_inst, df_edges, allowed_categories=["active_university", "closed_2016_khk", "closed_2020_cbk"])

print(f"Scope B1 Düğüm Sayısı: {G_b1.number_of_nodes()}, Kenar Sayısı: {G_b1.number_of_edges()}")
print(f"Scope B  Düğüm Sayısı: {G_b.number_of_nodes()}, Kenar Sayısı: {G_b.number_of_edges()}")

Scope B1 Düğüm Sayısı: 186, Kenar Sayısı: 3848
Scope B  Düğüm Sayısı: 194, Kenar Sayısı: 3887


## 2. Temel Ağ Ölçüleri ve Merkeziyet Analizleri
- Düğüm ve kenar sayıları, yoğunluk, kümelenme katsayısı, derece benzeşimi.
- Bağlantısız ağlarda tüm ağ ile dev bağlı bileşen (LCC) ayrımı.
- Degree, Weighted Degree (Strength), Betweenness, Closeness ve PageRank merkeziyetleri ile Spearman korelasyonu.

In [3]:
topo_b1 = compute_network_topology(G_b1, scope_name="Scope B1")
topo_b = compute_network_topology(G_b, scope_name="Scope B")

df_topo = pd.DataFrame([
    {"Metrik": "Düğüm Sayısı (N)", "Scope B1 (Aktif Üniversiteler)": topo_b1["number_of_nodes"], "Scope B (Tüm Üniversiteler)": topo_b["number_of_nodes"]},
    {"Metrik": "Kenar Sayısı (M)", "Scope B1 (Aktif Üniversiteler)": topo_b1["number_of_edges"], "Scope B (Tüm Üniversiteler)": topo_b["number_of_edges"]},
    {"Metrik": "Ağ Yoğunluğu (Density)", "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["density"]:.4f}", "Scope B (Tüm Üniversiteler)": f"{topo_b["density"]:.4f}"},
    {"Metrik": "Bağlı Bileşen Sayısı", "Scope B1 (Aktif Üniversiteler)": topo_b1["number_connected_components"], "Scope B (Tüm Üniversiteler)": topo_b["number_connected_components"]},
    {"Metrik": "İzole Düğüm Sayısı", "Scope B1 (Aktif Üniversiteler)": topo_b1["number_isolated_nodes"], "Scope B (Tüm Üniversiteler)": topo_b["number_isolated_nodes"]},
    {"Metrik": "Ortalama Kümelenme (C)", "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["average_clustering_coefficient"]:.4f}", "Scope B (Tüm Üniversiteler)": f"{topo_b["average_clustering_coefficient"]:.4f}"},
    {"Metrik": "Derece Benzeşimi (r)", "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["degree_assortativity"]:.4f}", "Scope B (Tüm Üniversiteler)": f"{topo_b["degree_assortativity"]:.4f}"},
    {"Metrik": "Ortalama En Kısa Yol (Tüm Ağ)", "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["full_network_average_path_length"]:.4f}", "Scope B (Tüm Üniversiteler)": "Tanımsız (Bağlantısız Ağ)"},
    {"Metrik": "LCC Ortalama En Kısa Yol (L)", "Scope B1 (Aktif Üniversiteler)": f"{topo_b1["largest_connected_component"]["average_shortest_path_length"]:.4f}", "Scope B (Tüm Üniversiteler)": f"{topo_b["largest_connected_component"]["average_shortest_path_length"]:.4f}"},
    {"Metrik": "Ağ Çapı (LCC Diameter)", "Scope B1 (Aktif Üniversiteler)": topo_b1["largest_connected_component"]["diameter"], "Scope B (Tüm Üniversiteler)": topo_b["largest_connected_component"]["diameter"]}
])
display(df_topo)

Metrik,Scope B1 (Aktif Üniversiteler),Scope B (Tüm Üniversiteler)
Düğüm Sayısı (N),186,194
Kenar Sayısı (M),3848,3887
Ağ Yoğunluğu (Density),0.2237,0.2076
Bağlı Bileşen Sayısı,1,2
İzole Düğüm Sayısı,0,1
Ortalama Kümelenme (C),0.4433,0.4477
Derece Benzeşimi (r),-0.0992,-0.0983
Ortalama En Kısa Yol (Tüm Ağ),1.8290,Tanımsız (Bağlantısız Ağ)
LCC Ortalama En Kısa Yol (L),1.8290,1.8731
Ağ Çapı (LCC Diameter),4,4


In [4]:
centrality_df, top10_df, corr_matrix = compute_centralities(G_b1, inst_metadata_df=df_b1)

print("--- İlk 10 Kurum Karşılaştırması (Derece Merkeziliği) ---")
deg_top10 = top10_df[top10_df["metric"] == "degree"].copy()
display(deg_top10[["rank", "display_name", "city", "metric_value", "period_cs_works_count"]].rename(columns={"metric_value": "degree"}))

print("\n--- Merkeziyetler ve CS Yayın Sayısı Arası Spearman Sıra Korelasyonu ---")
display(corr_matrix.round(4))

--- İlk 10 Kurum Karşılaştırması (Derece Merkeziliği) ---


rank,display_name,city,degree,weighted_degree,period_cs_works_count
1,Gazi University,Ankara,119.0,580.0,1170
2,Hacettepe University,Ankara,115.0,587.0,1019
3,Yıldız Technical University,Istanbul,113.0,542.0,1043
4,Istanbul University,Istanbul,111.0,744.0,712
5,Istanbul Technical University,Istanbul,105.0,564.0,1670
6,Ankara University,Ankara,105.0,439.0,700
7,Istanbul University-Cerrahpaşa,Istanbul,94.0,602.0,480
8,Karadeniz Technical University,Trabzon,93.0,323.0,573
9,Fırat University,Elâzığ,90.0,506.0,1173
10,Marmara University,Istanbul,90.0,344.0,636



--- Merkeziyetler ve CS Yayın Sayısı Arası Spearman Sıra Korelasyonu ---


,degree,weighted_degree,betweenness,closeness,pagerank,period_cs_works
degree,1.0000,0.9129,0.9054,0.9969,0.9042,0.8565
weighted_degree,0.9129,1.0000,0.8423,0.9126,0.9892,0.9043
betweenness,0.9054,0.8423,1.0000,0.9099,0.8818,0.7582
closeness,0.9969,0.9126,0.9099,1.0000,0.9062,0.8549
pagerank,0.9042,0.9892,0.8818,0.9062,1.0000,0.8834
period_cs_works,0.8565,0.9043,0.7582,0.8549,0.8834,1.0000


## 3. Derece Dağılımı ve Güç Yasası (Power Law) Analizi
- `powerlaw` kütüphanesi ile kesikli güç yasası fiti yapılır ($\alpha=2.90, x_{min}=29.0$).
- Üstel ve log-normal modellerle log-likelihood ratio testleri yürütülür.
- **Bilimsel Sınırlılık:** $N=186$ gibi dar derece aralığına sahip ağlarda saf bir güç yasası (scale-free) olduğunu iddia etmek bilimsel olarak yanıltıcıdır; dağılım log-normal modeli tercih etmektedir ($R=-3.54, p=0.0004$).

In [5]:
pw_results_path = os.path.join(REPORTS_GEN_DIR, "powerlaw_fit_results.json")
with open(pw_results_path, "r", encoding="utf-8") as f:
    pw_res = json.load(f)

print(f"Power Law Alpha: {pw_res["alpha"]:.4f}")
print(f"Minimum Eşik (xmin): {pw_res["xmin"]}")
print(f"Standart Hata (sigma): {pw_res["sigma"]:.4f}")
print(f"Üstel Modele Karşı R: {pw_res["comparison"]["vs_exponential"]["R"]:.3f} (p={pw_res["comparison"]["vs_exponential"]["p_value"]:.2e}) -> favors: {pw_res["comparison"]["vs_exponential"]["favors"]}")
print(f"Log-Normal Modele Karşı R: {pw_res["comparison"]["vs_lognormal"]["R"]:.3f} (p={pw_res["comparison"]["vs_lognormal"]["p_value"]:.4f}) -> favors: {pw_res["comparison"]["vs_lognormal"]["favors"]}")
print(f"Saf Güç Yasası Destekleniyor mu?: {pw_res["comparison"]["supports_pure_power_law"]}")

# Grafiği göster
Image(os.path.join(REPORTS_GEN_DIR, "degree_distribution_powerlaw.png"))

Power Law Alpha: 2.8989
Minimum Eşik (xmin): 29.0
Standart Hata (sigma): 0.1653
Üstel Modele Karşı R: -5.140 (p=2.74e-07) -> favors: exponential
Log-Normal Modele Karşı R: -3.537 (p=0.0004) -> favors: lognormal
Saf Güç Yasası Destekleniyor mu?: False


## 4. Topluluk Tespiti: Louvain ve Leiden Kararlılık Analizi
- Her iki algoritma da aynı ağırlıklı NetworkX grafiği ve `nx.community.modularity(..., weight='weight')` ortak ölçümüyle karşılaştırılmıştır.
- 10 farklı sabit tohum (`SEEDS_10`) ile NMI ve ARI kararlılık analizi yürütülmüştür.
- Leiden'ın kendi ModularityVertexPartition ham değeri ayrıca raporda saklanmıştır.

In [6]:
comm_summary_path = os.path.join(REPORTS_GEN_DIR, "community_detection_summary.json")
with open(comm_summary_path, "r", encoding="utf-8") as f:
    comm_sum = json.load(f)

print(f"Louvain Ortalama Topluluk Sayısı: {comm_sum["louvain"]["num_communities_mean"]:.1f}, Ortak Modülerlik (Q): {comm_sum["louvain"]["modularity_mean"]:.4f} ± {comm_sum["louvain"]["modularity_std"]:.4f}")
print(f"Louvain Kararlılık: NMI={comm_sum["louvain"]["stability"]["nmi_mean"]:.4f}, ARI={comm_sum["louvain"]["stability"]["ari_mean"]:.4f}")
print(f"\nLeiden  Ortalama Topluluk Sayısı: {comm_sum["leiden"]["num_communities_mean"]:.1f}, Ortak Modülerlik (Q): {comm_sum["leiden"]["modularity_mean"]:.4f} ± {comm_sum["leiden"]["modularity_std"]:.4f}")
print(f"Leiden  Ham Partition Modülerliği: {comm_sum["leiden"]["raw_partition_modularity_mean"]:.4f} ± {comm_sum["leiden"]["raw_partition_modularity_std"]:.4f}")
print(f"Leiden  Kararlılık: NMI={comm_sum["leiden"]["stability"]["nmi_mean"]:.4f}, ARI={comm_sum["leiden"]["stability"]["ari_mean"]:.4f}")
print(f"\nİki Algoritma Arası Çapraz Mutabakat: NMI={comm_sum["cross_algorithm_agreement"]["nmi"]:.4f}, ARI={comm_sum["cross_algorithm_agreement"]["ari"]:.4f}")

df_top_clusters = pd.read_csv(os.path.join(REPORTS_GEN_DIR, "community_top_clusters.csv"))
print("\n--- En Büyük Topluluklar ve Merkez (Hub) Kurumları (Seed=42) ---")
display(df_top_clusters[["community_id", "size", "ratio", "internal_edge_weight", "egean_institutions_count", "top_hub_institutions"]])

Louvain Ortalama Topluluk Sayısı: 6.5, Ortak Modülerlik (Q): 0.3394 ± 0.0015
Louvain Kararlılık: NMI=0.8171, ARI=0.8032

Leiden  Ortalama Topluluk Sayısı: 6.6, Ortak Modülerlik (Q): 0.3420 ± 0.0009
Leiden  Ham Partition Modülerliği: 0.1416 ± 0.0025
Leiden  Kararlılık: NMI=0.8378, ARI=0.8374

İki Algoritma Arası Çapraz Mutabakat: NMI=0.7845, ARI=0.7589

--- En Büyük Topluluklar ve Merkez (Hub) Kurumları (Seed=42) ---


community_id,size,ratio,internal_edge_weight,egean_institutions_count,top_hub_institutions
0,43,0.2312,1412,2,"Gazi University, Ankara University, Middle East Technical University"
1,41,0.2204,1546,3,"Istanbul University, Istanbul University-Cerrahpaşa, Marmara University"
2,37,0.1989,1458,2,"Hacettepe University, Yıldız Technical University, Istanbul Technical University"
3,31,0.1667,798,1,"Fırat University, Erciyes University, Atatürk University"
4,28,0.1505,674,4,"Karadeniz Technical University, Ege University, Dokuz Eylül University"


## 5. Null Model Analizi (100 Derece Korumalı Rastgele Ağ)
> **Yeniden Üretilebilirlik Açıklaması:**  
> Aşağıdaki hücre, `scripts/run_final_analysis.py` tarafından üretilen 100 rastgele ağlık null model sonuçlarını (`null_model_comparison.json` ve `null_model_comparison.csv`) okuyarak sunar. 100 iterasyonluk monte carlo deneyinin tamamını yeniden koşturmak için komut satırından `python scripts/run_final_analysis.py` çalıştırılmalıdır.

- **Metodoloji:** Gözlenen ağın kenar ağırlıklarını taşımayan basit ağırlıksız topolojik grafiği üzerinden `connected_double_edge_swap` ile derece dizisi %100 korunan 100 rastgele ağ üretilmiştir.
- **Modülerlik:** Hem gözlenen ağda hem rastgele ağlarda Louvain modülerliği açıkça `weight=None` ile hesaplanmıştır.
- **Ampirik p-Değeri:** Kümelenme ve modülerlik için sağ kuyruklu ($H_1: obs > null$), en kısa yol için iki kuyruklu ($H_1: obs \neq null$) Monte Carlo test formülü ($p = [r + 1]/[B + 1]$) uygulanmıştır.

In [7]:
null_summary_path = os.path.join(REPORTS_GEN_DIR, "null_model_comparison.json")
with open(null_summary_path, "r", encoding="utf-8") as f:
    null_sum = json.load(f)

df_null = pd.read_csv(os.path.join(REPORTS_GEN_DIR, "null_model_comparison.csv"))
display(df_null.round(4))

gamma = null_sum["small_worldness"]["gamma (C/C_null)"]
lambda_val = null_sum["small_worldness"]["lambda (L/L_null)"]
sigma = null_sum["small_worldness"]["sigma"]
print(f"\nKüçük Dünya (Small-Worldness) Katsayısı:")
print(f"  gamma (C / C_null) = {gamma:.4f}")
print(f"  lambda (L / L_null) = {lambda_val:.4f}")
print(f"  sigma = gamma / lambda = {sigma:.4f} > 1.0 (Küçük Dünya Özelliği Doğrulandı: {null_sum["small_worldness"]["is_small_world"]})")

metric,observed,null_mean,null_std,z_score,empirical_p_value,test_direction
Average Clustering Coefficient (C),0.4433,0.3898,0.0045,11.9977,0.0099,"right-tailed (H1: obs > null, p = [count(null >= obs) + 1] / [B + 1])"
Average Shortest Path Length (L),1.8290,1.8079,0.0035,6.0465,0.0198,"two-tailed (H1: obs != null, p = min(1.0, 2 * min(p_left, p_right)))"
Louvain Modularity (Q),0.1684,0.0962,0.0031,23.0494,0.0099,"right-tailed (H1: obs > null, p = [count(null >= obs) + 1] / [B + 1])"



Küçük Dünya (Small-Worldness) Katsayısı:
  gamma (C / C_null) = 1.1373
  lambda (L / L_null) = 1.0117
  sigma = gamma / lambda = 1.1242 > 1.0 (Küçük Dünya Özelliği Doğrulandı: True)


## 6. Zorunlu İleri Yöntem: Çekim (Gravity) Modeli Regresyonu
> **Yeniden Üretilebilirlik Açıklaması:**  
> Bu hücre, $\\binom{186}{2} = 17.205$ ikili üniversite çifti üzerinden hesaplanan Poisson ve veriden serbest $\\alpha$ tahmin eden Negatif Binom regresyon sonuçlarını (`gravity_model_results.json` ve `gravity_model_summary.csv`) inceler.  
> Model eğitimi ve ikili veri seti üretimini sıfırdan koşturmak için ana giriş noktası `scripts/run_final_analysis.py` betiğidir.

- **Mesafe Esnekliği:** Bağımsız değişken $\\ln(\\text{dist}_{eff})$ olduğundan $\\beta = -0.3010$ bir esneklik (elasticity) katsayısıdır. Coğrafi mesafe %1 arttığında iki üniversite arasındaki iş birliği beklentisi yaklaşık %0.30 azalmaktadır (kilometre başına doğrusal azalış değildir).
- **Aşırı Yayılım:** Poisson modelinde Pearson dispersiyon oranı 4.67 bulunarak aşırı yayılım kanıtlanmış, veriden tahmin edilen $\\alpha = 2.6511$ ($p < 10^{-248}$) parametresiyle Negative Binomial (NB2) esas model olarak seçilmiştir (AIC: Poisson 38.293, Negatif Binom 27.700).
- **Ege Bölgesi Etkisi:** $BothEgean$ kukla değişkeni ($\\beta=0.7899, p < 0.001, \\exp(\\beta)=2.203$), kurum büyüklüğü ve mesafe sabit tutulduğunda her iki kurumun da Ege'de olmasının iş birliği hacmini yaklaşık 2.2 katına çıkardığını göstermektedir.

In [8]:
grav_results_path = os.path.join(REPORTS_GEN_DIR, "gravity_model_results.json")
with open(grav_results_path, "r", encoding="utf-8") as f:
    grav_results = json.load(f)

df_grav_summary = pd.read_csv(os.path.join(REPORTS_GEN_DIR, "gravity_model_summary.csv"))
display(df_grav_summary.round(4))

synth = grav_results["synthesis"]
print("\n--- Aşırı Yayılım ve Esas Model Seçimi ---")
print(f"Pearson Dispersiyon Oranı: {grav_results["poisson_model"]["pearson_dispersion_ratio"]:.2f}")
print(f"Aşırı Yayılım Tespit Edildi mi?: {grav_results["poisson_model"]["overdispersion_detected"]}")
print(f"Esas Model: {synth["primary_model"]}")
print(f"Tahmin Edilen Alfa (Aşırı Yayılım) Parametresi: {grav_results["negative_binomial_model"]["alpha_parameter"]["estimated_alpha"]:.4f} (p < 0.001)")

print("\n--- Bilimsel Yorumlar (Esas Model Katsayılarından Üretilmiştir) ---")
print(f"Mesafe Esnekliği: {synth["scientific_conclusion"]["distance_elasticity"]["interpretation"]}")
print(f"Ege Bölgesi Çekim Etkisi: {synth["scientific_conclusion"]["regional_egean_effect"]["interpretation"]}")
print(f"Nedensellik Uyarısı: {synth["scientific_conclusion"]["causality_warning"]}")

term,poisson_coef,poisson_se,poisson_p,nb_coef,nb_se,nb_p,nb_exp_coef
Intercept,-7.1920,0.0898,0.0,-7.0893,0.1712,0.0000,0.0008
log_size_i,0.7483,0.0109,0.0,0.7390,0.0211,0.0000,2.0938
log_size_j,0.7567,0.0102,0.0,0.7780,0.0204,0.0000,2.1771
log_distance,-0.2672,0.0034,0.0,-0.3010,0.0100,0.0000,0.7401
both_egean,0.8458,0.0611,0.0,0.7899,0.1963,0.0001,2.2031
alpha,NaN,NaN,NaN,2.6511,0.0786,0.0000,14.1690



--- Aşırı Yayılım ve Esas Model Seçimi ---
Pearson Dispersiyon Oranı: 4.67
Aşırı Yayılım Tespit Edildi mi?: True
Esas Model: Negative Binomial (NB2)
Tahmin Edilen Alfa (Aşırı Yayılım) Parametresi: 2.6511 (p < 0.001)

--- Bilimsel Yorumlar (Esas Model Katsayılarından Üretilmiştir) ---
Mesafe Esnekliği: Model log-mesafe içerdiğinden β=-0.3010 (p < 0.001, %95 GA: [-0.3205, -0.2814]) katsayısı bir esneklik (elasticity) parametresidir; kilometre başına doğrusal etki değildir. Coğrafi mesafe %1 arttığında iki üniversite arasındaki beklenen iş birliği yaklaşık %0.30 azalmaktadır (mesafe iki katına çıktığında ortak yayın beklentisi 2^(-0.3010) ≈ 0.812 katına, yani %18.8 düzeyine inmektedir).
Ege Bölgesi Çekim Etkisi: Her iki kurumun da Ege Bölgesi'nde yer alması iş birliğini pozitif ve istatistiksel olarak anlamlı biçimde artırmaktadır (β=0.7899, %95 GA: [0.4051, 1.1747], p=5.74e-05, exp(β)=2.203). Diğer koşullar (kurum büyüklüğü ve aradaki coğrafi mesafe) eşit tutulduğunda, her iki kurumun d

## 7. Türkiye Coğrafi İş Birliği Haritası
Üretilen etkileşimli harita `reports/generated/turkey_collaboration_map.html` dosyası olarak kaydedilmiştir. Folium tabanlı bu haritada şehir içi koordinat çakışmaları açısal dağıtım (jitter) ile çözülmüş, düğüm boyutları derece ile ölçeklendirilmiş ve güçlü iş birlikleri ($w \ge 5$) görselleştirilmiştir.

In [9]:
map_file_path = os.path.join(REPORTS_GEN_DIR, "turkey_collaboration_map.html")
print("Harita dosya yolu: reports/generated/turkey_collaboration_map.html")
print(f"Harita dosya boyutu: {os.path.getsize(map_file_path)} bayt")

# Iframe ile haritayı göm
HTML(f'<iframe src="../reports/generated/turkey_collaboration_map.html" width="100%" height="500px"></iframe>')

Harita dosya yolu: reports/generated/turkey_collaboration_map.html
Harita dosya boyutu: 1007999 bayt
